In [1]:
import os
from paths import ensure_hf_file
import pickle

cal_path = "../datasets/deepreview_13k_calibration"
review_path = "../results/bench_reviews"


emb_path = ensure_hf_file("human_reviews_embeddings_deepreview.pkl")
idx_path = ensure_hf_file("human_review_score_index_deepreview.pkl")


with open(emb_path, "rb") as f:
    review_embeddings = pickle.load(f)

with open(idx_path, "rb") as f:
    review_score_index = pickle.load(f)


In [2]:
os.environ["ANTHROPIC_API_KEY"] = ""

In [3]:
import numpy as np
reviews = os.listdir(review_path)

bin_names = ["very_low", "low", "medium", "high", "very_high"]
thresholds = [2, 4, 6, 8]

bins = {name: [] for name in bin_names}
bins_embeddings = {name: [] for name in bin_names}

for i in review_score_index:
    score = review_score_index[i]
    idx = next((j for j, t in enumerate(thresholds) if score <= t), len(thresholds))
    name = bin_names[idx]
    bins[name].append(i)
    bins_embeddings[name].append(review_embeddings[i])

for name in bin_names:
    bins_embeddings[name] = np.array(bins_embeddings[name])

In [4]:
from openai import OpenAI
or_client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))

In [5]:
prompt = """
You are given a paper review and a set of calibration reviews for comparison. For each calibration review, assess how the given review differs in stance, severity, and reasoning. Then produce a final estimated score for the given review on this scale:

1 - strong reject
3 - reject
4 - borderline reject
6 - borderline accept
8 - accept
10 - strong accept

Score must be between 1 and 10 in increments of 0.5.

Given Review:
{review_content}
Calibration Reviews (under `/home/wg25r/split_review/datasets/deepreview_13k_calibration`):
{calibration_reviews}
"""

In [ ]:
import asyncio
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock

async def cal_paper(review):
    with open(os.path.join(review_path, review), "r") as f:
        review_content = f.read().split("<score>")[0].replace("MY FINAL SCORE:", "").strip()

    query_embedding = or_client.embeddings.create(
        model="google/gemini-embedding-001",
        input=review_content,
        encoding_format="float",
    )
    query_vector = np.array(query_embedding.data[0].embedding)

    selected_names = []
    for name in bin_names:
        if len(bins_embeddings[name]) == 0:
            continue
        similarities = bins_embeddings[name] @ query_vector.T
        top_indices = np.argsort(similarities)[-2:]
        selected_names.extend(bins[name][idx] for idx in top_indices)

    print(review_content)
    async for message in query(
        prompt=prompt.format(
            review_content=review_content,
            calibration_reviews="\n".join(selected_names),
        ),
        options=ClaudeAgentOptions(allowed_tools=["Read", "Edit", "Bash"]),
    ):
        if isinstance(message, AssistantMessage):
            for block in message.content:
                if isinstance(block, TextBlock):
                    print(block.text)

In [ ]:
for review in reviews: 
    await cal_paper(review)